# 中证800 V82 full_v46 生产级模型训练导出器

定位：只负责数据校验、训练计划生成、full_v46/defensive/balanced 模型训练、pkl 导出和完整性检查，不做策略筛选和回测实验。

默认导出：

- `full_v46 + expanding_min36 + legacy_rebalance`，cutoff: 2023/2024/2025；
- `full_v46 + rolling60m + legacy_rebalance`，cutoff: 2023/2024/2025。

如果要做严格标签边界导出，把 `LABEL_BOUNDARY_MODES` 改成 `['legacy_rebalance', 'label_end_safe']`。
数据更新：如需生成最新训练面板，在配置区把 `REBUILD_DATA=True`，必要时设置 `REBUILD_DATA_END_FOR_LABEL`。重建会根据当前 `RUN_FAMILIES` 抓取所需 raw jqfactor 字段，并在加载后生成 `irank__` 行业相对特征。

默认 `RUN_FAMILIES=["full_v46"]`；如需导出 defensive/balanced，把配置改成对应 family 或三组一起跑。


## 0. 导入与进度条


In [ ]:
import os
import gc
import json
import pickle
import shutil
import hashlib
import warnings
import builtins as _bi
import datetime as _dt
from pathlib import Path
from datetime import datetime

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception:
    pass

import lightgbm as lgb
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 120)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


## 1. 生产配置


In [ ]:
# =========================
# Product config
# =========================
PROJECT_DIR = Path.cwd()
PRODUCT_VERSION = "v82"
RUN_NAME = "full_v46_production_model_trainer"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = PROJECT_DIR / "csi800_ml_v82_full_v46_model_trainer_outputs"
MODEL_EXPORT_DIR = OUT_DIR / "model_exports"
MANIFEST_DIR = OUT_DIR / "manifests"
QA_DIR = OUT_DIR / "qa"
FIGURE_DIR = OUT_DIR / "figures"
for _p in [OUT_DIR, MODEL_EXPORT_DIR, MANIFEST_DIR, QA_DIR, FIGURE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

# Optional data rebuild. Keep False for normal model training.
# Set REBUILD_DATA=True inside JoinQuant research when you need to regenerate the CSI800 monthly training panel.
REBUILD_DATA = False
USE_REBUILT_DATA_FOR_TRAINING = True
REBUILD_FORCE_OVERWRITE = False
REBUILD_DATA_START = "2019-01-01"
# None means use today at runtime. For reproducible runs, set an explicit date such as "2026-07-10".
REBUILD_DATA_END_FOR_LABEL = None
REBUILD_PRICE_CHUNK_SIZE = 160
REBUILD_FACTOR_CHUNK_SIZE = 20
REBUILD_MIN_LISTING_DAYS = 180
REBUILD_UNIVERSE_NAME = "CSI800"
REBUILD_UNIVERSE_INDEX = "000906.XSHG"
_REBUILD_END_RESOLVED = REBUILD_DATA_END_FOR_LABEL or datetime.now().strftime("%Y-%m-%d")
REBUILD_DATA_TAG = "%s_%s" % (REBUILD_DATA_START.replace("-", ""), _REBUILD_END_RESOLVED.replace("-", ""))
REBUILD_DATA_OUTPUT_FILE = "train_csi800_factor_v40_data_enhancement_%s.csv" % REBUILD_DATA_TAG
REBUILD_DATA_OUTPUT_PATH = PROJECT_DIR / REBUILD_DATA_OUTPUT_FILE
REBUILD_MANIFEST_PATH = MANIFEST_DIR / ("v82_rebuild_data_manifest_%s.csv" % REBUILD_DATA_TAG)
DATA_CANDIDATES = [REBUILD_DATA_OUTPUT_PATH] + DATA_CANDIDATES

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
FACTOR_DATE_COL = "feature_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"

MODEL_FAMILY = "full_v46"
MODEL_ROLE = "production_candidate"
MODEL_VERSION_PREFIX = "v82"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
MIN_TRAIN_MONTHS = 36
PICKLE_PROTOCOL = 2

TOP_N_CANDIDATES = 30
STOCK_NUM = 8
PORTFOLIO_RULE = "top8_board_cap"
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])
INDUSTRY_CAP_RATIO = 0.20

# 默认只导出 V46 对齐口径；严格生产审计可加上 label_end_safe。
LABEL_BOUNDARY_MODES = ["legacy_rebalance"]
# LABEL_BOUNDARY_MODES = ["legacy_rebalance", "label_end_safe"]

TRAIN_CUTOFFS = ["2023-12-31", "2024-12-31", "2025-12-31"]
TRAIN_POLICIES = [
    {"train_policy": "expanding_min36", "method_type": "expanding", "train_window_months": None, "min_train_months": 36, "description": "expand from 2019-01"},
    {"train_policy": "rolling60m", "method_type": "rolling", "train_window_months": 60, "min_train_months": 60, "description": "latest 60 rebalance months"},
]
RUN_TRAIN_POLICIES = None  # example: ["expanding_min36"]
RUN_CUTOFFS = None         # example: ["2025-12-31"]
RUN_FAMILIES = ["full_v46"]  # examples: ["defensive_reversal_quality"], ["balanced_style_plus"], or all three


EXPORT_MODELS = True
RELOAD_CHECK = True
FAIL_ON_MISSING_FULL_FEATURES = True
FAIL_ON_MISSING_FAMILY_FEATURES = True
MIN_FEATURE_NON_NULL_RATIO = 0.01
SMOKE_TEST = False
SMOKE_MAX_TASKS = 1

# Honest temporal holdout diagnostics. The production model is still refit on all eligible rows.
ENABLE_HOLDOUT_DIAGNOSTICS = True
HOLDOUT_TRUE_TOP_N = 20
HOLDOUT_PRED_TOP_K = 8
PLOT_DPI = 140

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

if RUN_TRAIN_POLICIES is not None:
    _allow = set(RUN_TRAIN_POLICIES)
    TRAIN_POLICIES = [x for x in TRAIN_POLICIES if x["train_policy"] in _allow]
if RUN_CUTOFFS is not None:
    _cut = set([str(x) for x in RUN_CUTOFFS])
    TRAIN_CUTOFFS = [x for x in TRAIN_CUTOFFS if str(x) in _cut]

print("OUT_DIR:", OUT_DIR)
print("MODEL_EXPORT_DIR:", MODEL_EXPORT_DIR)
print("LABEL_BOUNDARY_MODES:", LABEL_BOUNDARY_MODES)
print("TRAIN_POLICIES:", [x["train_policy"] for x in TRAIN_POLICIES])
print("TRAIN_CUTOFFS:", TRAIN_CUTOFFS)
print("RUN_FAMILIES:", RUN_FAMILIES)
print("fixed_iter:", FIXED_ITER, "seed:", SEED)
print("REBUILD_DATA:", REBUILD_DATA, "REBUILD_DATA_OUTPUT_PATH:", REBUILD_DATA_OUTPUT_PATH)


## 2. full_v46 特征定义与 LGB 参数


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_V46_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)

V4_PRICE_PATH_COLS = [
    "px_ret_5", "px_ret_20", "px_ret_60", "px_ret_120",
    "px_close_to_ma20", "px_close_to_ma60", "px_ma20_to_ma60",
    "px_volatility_20", "px_volatility_60", "px_drawdown_20", "px_drawdown_60", "px_drawdown_120",
    "px_up_day_ratio_20", "px_new_high_distance_60", "px_new_low_distance_60",
    "px_skew_20", "px_kurt_20",
]
TRADE_LIQUIDITY_COLS = [
    "liq_money_mean_20", "liq_money_mean_60", "liq_money_ratio_20_60",
    "liq_volume_mean_20", "liq_volume_ratio_20_60",
    "liq_amplitude_mean_20", "liq_amplitude_mean_60",
    "liq_paused_count_20", "liq_paused_count_60",
    "liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20", "liq_one_price_limit_count_20",
]
CONTEXT_COLS = [
    "ctx_industry_ret_20", "ctx_industry_ret_60",
    "ctx_stock_minus_industry_ret_20", "ctx_stock_minus_industry_ret_60",
    "ctx_stock_rank_industry_ret_20", "ctx_stock_rank_industry_volatility_20",
    "ctx_market_ret_20", "ctx_market_ret_60", "ctx_market_volatility_20",
]
CORE_TEMPORAL_FACTORS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio",
    "Rank1M", "sharpe_ratio_60", "VOSC", "MFI14",
]

DERIVED_PREFIX = "irank__"


def industry_relative_name(col):
    return DERIVED_PREFIX + col


def is_industry_relative_feature(col):
    return isinstance(col, str) and col.startswith(DERIVED_PREFIX)


def add_industry_relative(cols):
    return [industry_relative_name(c) for c in cols]


BALANCED_STYLE_IRANK_BASE = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio", "roe_ttm", "roa_ttm",
    "growth", "Rank1M", "Price3M", "residual_volatility", "liquidity",
]
DEFENSIVE_REVERSAL_IRANK_BASE = [
    "roe_ttm", "net_operating_cash_flow_coverage", "debt_to_asset_ratio", "book_to_price_ratio",
    "cash_flow_to_price_ratio", "residual_volatility", "Variance20", "BIAS60", "CCI20", "liquidity",
]

BALANCED_STYLE_PLUS = unique_keep_order([
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "roe_ttm", "roa_ttm", "gross_profit_ttm",
    "net_operating_cash_flow_coverage", "operating_profit_to_total_profit", "adjusted_profit_to_total_profit",
    "growth", "operating_revenue_growth_rate", "net_profit_growth_rate",
    "size", "non_linear_size", "beta", "residual_volatility", "liquidity", "leverage", "momentum",
    "Rank1M", "Price3M", "ROC60", "MAC60", "MFI14", "px_close_to_ma60", "px_drawdown_60",
    "Variance20", "sharpe_ratio_60", "ATR6", "DAVOL10", "VOL10", "liq_money_ratio_20_60",
] + add_industry_relative(BALANCED_STYLE_IRANK_BASE))

DEFENSIVE_REVERSAL_QUALITY = unique_keep_order([
    "roe_ttm", "roa_ttm", "net_operating_cash_flow_coverage", "net_operate_cash_flow_to_total_liability",
    "super_quick_ratio", "current_ratio", "quick_ratio", "debt_to_asset_ratio", "debt_to_equity_ratio",
    "equity_to_asset_ratio", "ACCA",
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "cash_earnings_to_price_ratio",
    "residual_volatility", "Variance20", "Variance60", "Skewness20", "Skewness60",
    "Kurtosis20", "sharpe_ratio_20", "sharpe_ratio_60", "ATR6", "ATR14",
    "BIAS20", "BIAS60", "CCI20", "CCI88", "boll_down", "boll_up",
    "arron_up_25", "arron_down_25", "px_drawdown_60",
    "liquidity", "VOL20", "DAVOL20", "turnover_volatility",
] + add_industry_relative(DEFENSIVE_REVERSAL_IRANK_BASE))

ALL_FEATURE_FAMILIES = [
    {
        "family": "full_v46",
        "feature_variant": "full",
        "description": "V46/V61 full hybrid-light feature stack",
        "candidate_cols": FULL_V46_COLS,
        "min_available_cols": 20,
    },
    {
        "family": "defensive_reversal_quality",
        "feature_variant": "defensive",
        "description": "defensive quality, solvency, low-volatility shape and reversal/exhaustion chain",
        "candidate_cols": DEFENSIVE_REVERSAL_QUALITY,
        "min_available_cols": 20,
    },
    {
        "family": "balanced_style_plus",
        "feature_variant": "balanced",
        "description": "balanced valuation, quality, growth, style, trend and risk/liquidity chain",
        "candidate_cols": BALANCED_STYLE_PLUS,
        "min_available_cols": 20,
    },
]

if RUN_FAMILIES is None:
    FEATURE_FAMILIES = list(ALL_FEATURE_FAMILIES)
else:
    _run_family_set = set(RUN_FAMILIES)
    FEATURE_FAMILIES = [f for f in ALL_FEATURE_FAMILIES if f["family"] in _run_family_set]
if len(FEATURE_FAMILIES) == 0:
    raise ValueError("RUN_FAMILIES selected no valid family: %s" % RUN_FAMILIES)

ALL_CANDIDATE_COLS = []
for fam in FEATURE_FAMILIES:
    ALL_CANDIDATE_COLS.extend(fam["candidate_cols"])
ALL_CANDIDATE_COLS = unique_keep_order(ALL_CANDIDATE_COLS)
RAW_CANDIDATE_COLS = unique_keep_order([c for c in ALL_CANDIDATE_COLS if not is_industry_relative_feature(c)])
REBUILD_COMPUTED_COLS = unique_keep_order(V4_PRICE_PATH_COLS + TRADE_LIQUIDITY_COLS + CONTEXT_COLS + HYBRID_LIGHT_EXTRA_COLS + ["alpha_rank_pct"])
REBUILD_JQFACTOR_COLS = unique_keep_order([c for c in RAW_CANDIDATE_COLS if c not in REBUILD_COMPUTED_COLS])

feature_manifest_rows = []
for fam in FEATURE_FAMILIES:
    feature_manifest_rows.append({
        "family": fam["family"],
        "feature_variant": fam["feature_variant"],
        "candidate_feature_count": len(fam["candidate_cols"]),
        "description": fam["description"],
        "candidate_features": ",".join(fam["candidate_cols"]),
    })
feature_manifest_df = pd.DataFrame(feature_manifest_rows)
feature_manifest_df.to_csv(MANIFEST_DIR / "v82_feature_manifest.csv", index=False)
display_df(feature_manifest_df)


## 3. 可选数据重建模块

默认跳过。需要更新训练面板时，在 JoinQuant 研究环境把 `REBUILD_DATA=True`，本模块会重建 CSI800 月度因子/收益 CSV，再交给后续训练模块使用。


In [ ]:
# =========================
# Optional CSI800 V4/V46 data rebuild module
# =========================
def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def require_joinquant_api():
    try:
        get_trade_days(end_date="2019-01-02", count=1)
    except NameError:
        raise RuntimeError("REBUILD_DATA=True requires JoinQuant research runtime; get_trade_days is not available")
    except Exception:
        pass


def get_period_date(period, start_date, end_date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    if len(trade_days) == 0:
        return []
    if period != "M":
        raise ValueError("data rebuild only supports monthly period M")
    dates = []
    last_key = None
    for d in trade_days:
        key = d.strftime("%Y-%m")
        if key != last_key:
            dates.append(d.strftime("%Y-%m-%d"))
            last_key = key
    return dates


def get_previous_trade_date(date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(end_date=date, count=2))
    if len(trade_days) < 2:
        return None
    return trade_days[-2].strftime("%Y-%m-%d")


def filter_listing_age(stocks, begin_date, n=REBUILD_MIN_LISTING_DAYS):
    out = []
    begin_dt = pd.Timestamp(begin_date).to_pydatetime()
    for stock in stocks:
        try:
            info = get_security_info(stock)
        except Exception:
            info = None
        if info is None:
            continue
        if info.start_date <= (begin_dt - _dt.timedelta(days=n)).date():
            out.append(stock)
    return out


def filter_paused_stock_by_date(stock_list, date):
    if len(stock_list) == 0:
        return []
    try:
        paused_df = get_price(
            stock_list,
            end_date=date,
            frequency="daily",
            fields=["paused"],
            count=1,
            skip_paused=False,
            panel=False,
            fill_paused=True,
        )
    except Exception:
        return stock_list
    if paused_df is None or paused_df.empty or "paused" not in paused_df.columns:
        return stock_list
    paused_map = paused_df.groupby("code")["paused"].last()
    return [s for s in stock_list if (s not in paused_map.index) or (not bool(paused_map.loc[s]))]


def get_stock_for_rebuild(stock_pool, feature_date):
    require_joinquant_api()
    if stock_pool == "CSI800":
        stock_list = get_index_stocks(REBUILD_UNIVERSE_INDEX, feature_date)
    elif stock_pool == "HS300":
        stock_list = get_index_stocks("000300.XSHG", feature_date)
    elif stock_pool == "ZZ1000":
        stock_list = get_index_stocks("000852.XSHG", feature_date)
    elif stock_pool == "A":
        stock_list = get_index_stocks("000985.XSHG", feature_date)
    else:
        raise ValueError("unsupported stock_pool: " + str(stock_pool))
    if len(stock_list) == 0:
        return []
    try:
        st_data = get_extras("is_st", stock_list, count=1, end_date=feature_date)
    except Exception:
        st_data = None
    if st_data is not None and len(st_data) > 0:
        st_row = st_data.iloc[0]
        stock_list = [s for s in stock_list if (s not in st_row.index) or pd.isnull(st_row[s]) or (not bool(st_row[s]))]
    stock_list = filter_paused_stock_by_date(stock_list, feature_date)
    stock_list = filter_listing_age(stock_list, feature_date, n=REBUILD_MIN_LISTING_DAYS)
    return stock_list


def get_industry_bucket_map_for_rebuild(stock_list, date):
    if len(stock_list) == 0:
        return {}
    try:
        industry_info = get_industry(stock_list, date=date)
    except Exception:
        return {s: "UNKNOWN" for s in stock_list}
    out = {}
    for stock in stock_list:
        info = industry_info.get(stock, {})
        bucket = None
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = info.get(key, None)
            if isinstance(sub, dict):
                bucket = sub.get("industry_code") or sub.get("industry_name")
                if bucket:
                    break
        out[stock] = bucket if bucket else "UNKNOWN"
    return out


def get_factor_data_for_rebuild(stock_list, date):
    if len(stock_list) == 0:
        return pd.DataFrame()
    df_factor = pd.DataFrame(index=stock_list)
    for fac_chunk in chunks(REBUILD_JQFACTOR_COLS, REBUILD_FACTOR_CHUNK_SIZE):
        try:
            factor_data = get_factor_values(securities=stock_list, factors=fac_chunk, count=1, end_date=date)
        except Exception:
            factor_data = None
        if factor_data is not None:
            for fac in fac_chunk:
                try:
                    if fac in factor_data:
                        df_factor[fac] = factor_data[fac].iloc[0, :]
                    else:
                        df_factor[fac] = np.nan
                except Exception:
                    df_factor[fac] = np.nan
        else:
            for fac in fac_chunk:
                try:
                    one = get_factor_values(securities=stock_list, factors=[fac], count=1, end_date=date)
                    if one is not None and fac in one:
                        df_factor[fac] = one[fac].iloc[0, :]
                    else:
                        df_factor[fac] = np.nan
                except Exception:
                    df_factor[fac] = np.nan
    return df_factor


def calc_ret(close_mat, days):
    if close_mat is None or close_mat.empty or len(close_mat) <= days:
        return pd.Series(index=close_mat.columns if close_mat is not None else [], dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[-days - 1] - 1


def calc_up_day_ratio(ret_mat, days):
    if ret_mat is None or ret_mat.empty:
        return pd.Series(dtype=float)
    return (ret_mat.tail(days) > 0).mean()


def calc_new_low_distance(close_mat, days):
    if close_mat is None or close_mat.empty:
        return pd.Series(dtype=float)
    return close_mat.iloc[-1] / close_mat.tail(days).min() - 1


def safe_group_rank_pct(df, group_col, value_col):
    def _rank_one(s):
        valid_count = int(s.notnull().sum())
        if valid_count <= 0:
            return pd.Series(np.nan, index=s.index)
        return s.rank(method="average") / float(valid_count)
    return df.groupby(group_col)[value_col].transform(_rank_one)


def get_price_path_and_liquidity_data(stock_list, date, lookback=121, chunk_size=None):
    if chunk_size is None:
        chunk_size = REBUILD_PRICE_CHUNK_SIZE
    cols = V4_PRICE_PATH_COLS + TRADE_LIQUIDITY_COLS[:9]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "volume", "money", "paused"],
                count=lookback,
                skip_paused=False,
                fq="pre",
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None
        if price_df is None or price_df.empty:
            out_all.append(out)
            continue
        for col in ["close", "high", "low", "volume", "money", "paused"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        volume_mat = price_df.pivot_table(index="time", columns="code", values="volume").sort_index()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        paused_mat = price_df.pivot_table(index="time", columns="code", values="paused").sort_index()
        ret_mat = close_mat.pct_change()
        last_close = close_mat.iloc[-1]
        ma20 = close_mat.tail(20).mean()
        ma60 = close_mat.tail(60).mean()
        money20 = money_mat.tail(20).mean()
        money60 = money_mat.tail(60).mean()
        volume20 = volume_mat.tail(20).mean()
        volume60 = volume_mat.tail(60).mean()
        out["px_ret_5"] = calc_ret(close_mat, 5)
        out["px_ret_20"] = calc_ret(close_mat, 20)
        out["px_ret_60"] = calc_ret(close_mat, 60)
        out["px_ret_120"] = calc_ret(close_mat, 120)
        out["px_close_to_ma20"] = last_close / ma20 - 1
        out["px_close_to_ma60"] = last_close / ma60 - 1
        out["px_ma20_to_ma60"] = ma20 / ma60 - 1
        out["px_volatility_20"] = ret_mat.tail(20).std()
        out["px_volatility_60"] = ret_mat.tail(60).std()
        out["px_drawdown_20"] = last_close / close_mat.tail(20).max() - 1
        out["px_drawdown_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_drawdown_120"] = last_close / close_mat.tail(120).max() - 1
        out["px_up_day_ratio_20"] = calc_up_day_ratio(ret_mat, 20)
        out["px_new_high_distance_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_new_low_distance_60"] = calc_new_low_distance(close_mat, 60)
        out["px_skew_20"] = ret_mat.tail(20).skew()
        out["px_kurt_20"] = ret_mat.tail(20).kurt()
        out["liq_money_mean_20"] = money20
        out["liq_money_mean_60"] = money60
        out["liq_money_ratio_20_60"] = money20 / money60 - 1
        out["liq_volume_mean_20"] = volume20
        out["liq_volume_ratio_20_60"] = volume20 / volume60 - 1
        out["liq_amplitude_mean_20"] = (high_mat.tail(20) / low_mat.tail(20) - 1).mean()
        out["liq_amplitude_mean_60"] = (high_mat.tail(60) / low_mat.tail(60) - 1).mean()
        out["liq_paused_count_20"] = paused_mat.tail(20).fillna(0).sum()
        out["liq_paused_count_60"] = paused_mat.tail(60).fillna(0).sum()
        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, close_mat, high_mat, low_mat, volume_mat, money_mat, paused_mat, ret_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_limit_state_data(stock_list, date, lookback=20, chunk_size=None):
    if chunk_size is None:
        chunk_size = REBUILD_PRICE_CHUNK_SIZE
    cols = ["liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20", "liq_one_price_limit_count_20"]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "money", "paused", "high_limit", "low_limit"],
                count=lookback,
                skip_paused=False,
                fq=None,
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None
        if price_df is None or price_df.empty:
            out_all.append(out)
            continue
        for col in ["close", "high", "low", "money", "paused", "high_limit", "low_limit"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        high_limit_mat = price_df.pivot_table(index="time", columns="code", values="high_limit").sort_index()
        low_limit_mat = price_df.pivot_table(index="time", columns="code", values="low_limit").sort_index()
        money_stack = money_mat.stack().dropna()
        money_q20 = money_stack.quantile(0.20) if len(money_stack) else np.nan
        out["liq_low_money_days_20"] = (money_mat.tail(20) < money_q20).sum() if not pd.isnull(money_q20) else np.nan
        limit_up = close_mat >= (high_limit_mat * 0.999)
        limit_down = close_mat <= (low_limit_mat * 1.001)
        one_price = (high_mat <= low_mat * 1.0001) & (limit_up | limit_down)
        out["liq_limit_up_count_20"] = limit_up.tail(20).sum()
        out["liq_limit_down_count_20"] = limit_down.tail(20).sum()
        out["liq_one_price_limit_count_20"] = one_price.tail(20).sum()
        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, money_mat, close_mat, high_mat, low_mat, high_limit_mat, low_limit_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_market_context(date, benchmark=BENCHMARK, lookback=61):
    out = {"ctx_market_ret_20": np.nan, "ctx_market_ret_60": np.nan, "ctx_market_volatility_20": np.nan}
    try:
        bench_df = get_price(benchmark, end_date=date, frequency="daily", fields=["close"], count=lookback, skip_paused=True, fq="pre")
    except Exception:
        bench_df = None
    if bench_df is None or bench_df.empty or "close" not in bench_df.columns:
        return out
    close = bench_df["close"].dropna()
    if len(close) > 20:
        out["ctx_market_ret_20"] = close.iloc[-1] / close.iloc[-21] - 1
        out["ctx_market_volatility_20"] = close.pct_change().tail(20).std()
    if len(close) > 60:
        out["ctx_market_ret_60"] = close.iloc[-1] / close.iloc[-61] - 1
    return out


def attach_industry_context(factor_data, market_context):
    out = factor_data.copy()
    for col, value in market_context.items():
        out[col] = value
    for ret_col, ctx_col in [("px_ret_20", "ctx_industry_ret_20"), ("px_ret_60", "ctx_industry_ret_60")]:
        out[ctx_col] = out.groupby("industry_bucket")[ret_col].transform("mean")
    out["ctx_stock_minus_industry_ret_20"] = out["px_ret_20"] - out["ctx_industry_ret_20"]
    out["ctx_stock_minus_industry_ret_60"] = out["px_ret_60"] - out["ctx_industry_ret_60"]
    out["ctx_stock_rank_industry_ret_20"] = safe_group_rank_pct(out, "industry_bucket", "px_ret_20")
    out["ctx_stock_rank_industry_volatility_20"] = safe_group_rank_pct(out, "industry_bucket", "px_volatility_20")
    return out


def get_forward_alpha(stock_list, date, next_date, benchmark):
    if len(stock_list) == 0:
        return pd.Series(dtype=float)
    price_df = get_price(stock_list, start_date=date, end_date=next_date, frequency="daily", fields=["close"], skip_paused=True, fq="pre", panel=False)
    if price_df is None or price_df.empty:
        return pd.Series(dtype=float)
    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
    if len(close_mat) < 2:
        return pd.Series(dtype=float)
    stock_ret = close_mat.iloc[-1] / close_mat.iloc[1] - 1
    bench_df = get_price(benchmark, start_date=date, end_date=next_date, frequency="daily", fields=["close"], skip_paused=True, fq="pre")
    if bench_df is None or bench_df.empty or len(bench_df) < 2:
        return pd.Series(dtype=float)
    bench_ret = bench_df["close"].iloc[-1] / bench_df["close"].iloc[1] - 1
    return stock_ret - bench_ret


def add_core_factor_temporal_features(df):
    out = df.copy().sort_values(["rebalance_date", "stock"]).reset_index(drop=True)
    for factor in CORE_TEMPORAL_FACTORS:
        if factor not in out.columns:
            continue
        rank_col = "tmp_%s_rank" % factor
        out[rank_col] = safe_group_rank_pct(out, "rebalance_date", factor)
        g_stock = out.groupby("stock")[rank_col]
        for lag in [1, 3]:
            out["ts_%s_rank_chg_%sm" % (factor, lag)] = out[rank_col] - g_stock.shift(lag)
        out["ts_%s_rank_mean_3m" % factor] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
        out["ts_%s_rank_std_3m" % factor] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
        rolling_mean_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
        rolling_std_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).std())
        out["ts_%s_rank_z_6m" % factor] = (out[rank_col] - rolling_mean_6) / rolling_std_6
        out = out.drop(columns=[rank_col])
    return out


def build_v46_rebuild_dataset():
    require_joinquant_api()
    date_list = get_period_date("M", REBUILD_DATA_START, _REBUILD_END_RESOLVED)
    print("rebuild rebalance dates =", len(date_list), "|", REBUILD_DATA_START, "->", _REBUILD_END_RESOLVED)
    all_rows = []
    loop = list(enumerate(date_list[:-1]))
    for i, rebalance_date in progress_iter(loop, total=len(loop), desc="rebuild monthly data"):
        next_date = date_list[i + 1]
        feature_date = get_previous_trade_date(rebalance_date)
        if feature_date is None:
            continue
        stock_list = get_stock_for_rebuild(REBUILD_UNIVERSE_NAME, feature_date)
        if len(stock_list) == 0:
            continue
        jq_factor_data = get_factor_data_for_rebuild(stock_list, feature_date)
        if jq_factor_data is None or jq_factor_data.empty:
            continue
        industry_map = get_industry_bucket_map_for_rebuild(stock_list, feature_date)
        price_liq_data = get_price_path_and_liquidity_data(stock_list, feature_date)
        limit_data = get_limit_state_data(stock_list, feature_date)
        alpha = get_forward_alpha(stock_list, rebalance_date, next_date, BENCHMARK)
        if alpha.empty:
            continue
        factor_data = jq_factor_data.join(price_liq_data, how="left").join(limit_data, how="left")
        factor_data["stock"] = factor_data.index
        factor_data["industry_bucket"] = factor_data["stock"].map(industry_map).fillna("UNKNOWN")
        factor_data = attach_industry_context(factor_data, get_market_context(feature_date, BENCHMARK))
        factor_data["alpha_1m"] = alpha
        factor_data["rebalance_date"] = rebalance_date
        factor_data["feature_date"] = feature_date
        factor_data["next_date"] = next_date
        factor_data = factor_data.dropna(subset=["alpha_1m"]).copy()
        if len(factor_data) < 30:
            continue
        factor_data["alpha_rank_pct"] = factor_data["alpha_1m"].rank(method="first") / float(len(factor_data))
        all_rows.append(factor_data.reset_index(drop=True))
        print("  rebuilt %s/%s rebalance=%s feature=%s rows=%s" % (i + 1, _bi.max(1, len(date_list) - 1), rebalance_date, feature_date, len(factor_data)))
        del jq_factor_data, price_liq_data, limit_data, alpha, factor_data
        gc.collect()
    if len(all_rows) == 0:
        raise ValueError("data rebuild produced no rows")
    df = pd.concat(all_rows, ignore_index=True)
    df = add_core_factor_temporal_features(df)
    df.to_csv(str(REBUILD_DATA_OUTPUT_PATH), index=False)
    print("rebuilt data rows =", len(df), "saved ->", REBUILD_DATA_OUTPUT_PATH)
    return df


def maybe_rebuild_dataset():
    global DATA_PATH_OVERRIDE
    if not REBUILD_DATA:
        print("skip data rebuild; set REBUILD_DATA=True to regenerate data")
        return None
    if REBUILD_DATA_OUTPUT_PATH.exists() and not REBUILD_FORCE_OVERWRITE:
        print("rebuilt data already exists; skip rebuild:", REBUILD_DATA_OUTPUT_PATH)
        if USE_REBUILT_DATA_FOR_TRAINING:
            DATA_PATH_OVERRIDE = str(REBUILD_DATA_OUTPUT_PATH)
            print("DATA_PATH_OVERRIDE switched to:", DATA_PATH_OVERRIDE)
        return pd.read_csv(str(REBUILD_DATA_OUTPUT_PATH), nrows=5)
    print("rebuilding CSI800 monthly training panel")
    print("  start =", REBUILD_DATA_START, "end_for_label =", _REBUILD_END_RESOLVED)
    rebuilt_df = build_v46_rebuild_dataset()
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col in rebuilt_df.columns:
            rebuilt_df[col] = pd.to_datetime(rebuilt_df[col]).dt.normalize()
    meta_cols = ["stock", "rebalance_date", "feature_date", "next_date", "alpha_1m", "alpha_rank_pct", "industry_bucket"]
    feature_cols_rebuilt = [c for c in rebuilt_df.columns if c not in meta_cols]
    date_min = rebuilt_df["rebalance_date"].min() if len(rebuilt_df) else pd.NaT
    date_max = rebuilt_df["rebalance_date"].max() if len(rebuilt_df) else pd.NaT
    rebuild_manifest = pd.DataFrame([{
        "data_file": str(REBUILD_DATA_OUTPUT_PATH),
        "rows": int(len(rebuilt_df)),
        "months": int(rebuilt_df["rebalance_date"].nunique()) if "rebalance_date" in rebuilt_df.columns else 0,
        "stock_count": int(rebuilt_df["stock"].nunique()) if "stock" in rebuilt_df.columns else 0,
        "rebalance_date_min": str(date_min.date()) if not pd.isnull(date_min) else "",
        "rebalance_date_max": str(date_max.date()) if not pd.isnull(date_max) else "",
        "feature_count": int(len(feature_cols_rebuilt)),
        "rebuild_jqfactor_count": int(len(REBUILD_JQFACTOR_COLS)),
        "rebuild_jqfactor_cols": ",".join(REBUILD_JQFACTOR_COLS),
        "full_v46_missing_features": ",".join([c for c in FULL_V46_COLS if c not in rebuilt_df.columns]),
        "rebuild_data_start": REBUILD_DATA_START,
        "rebuild_data_end_for_label": _REBUILD_END_RESOLVED,
        "universe": REBUILD_UNIVERSE_NAME,
        "universe_index": REBUILD_UNIVERSE_INDEX,
        "created_at": RUN_TIMESTAMP,
    }])
    rebuild_manifest.to_csv(str(REBUILD_MANIFEST_PATH), index=False)
    print("saved rebuild manifest ->", REBUILD_MANIFEST_PATH)
    if USE_REBUILT_DATA_FOR_TRAINING:
        DATA_PATH_OVERRIDE = str(REBUILD_DATA_OUTPUT_PATH)
        print("DATA_PATH_OVERRIDE switched to:", DATA_PATH_OVERRIDE)
    return rebuilt_df


## 4. 通用工具函数


In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))



def safe_mean(values):
    s = pd.Series(list(values), dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.mean()) if len(s) else np.nan


def binary_rank_auc(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "score": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3:
        return np.nan
    positive = d["y"] > 0
    n_pos = int(positive.sum())
    n_neg = int(len(d) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = d["score"].rank(method="average", ascending=True)
    rank_sum = float(ranks[positive].sum())
    return (rank_sum - n_pos * (n_pos + 1) / 2.0) / float(n_pos * n_neg)


def binary_ndcg_at_k(y_true, scores, k):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "score": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("score", ascending=False)
    k_eff = _bi.min(int(k), len(d))
    if k_eff <= 0:
        return np.nan
    gains = (d["y"].head(k_eff).values > 0).astype(float)
    discounts = np.log2(np.arange(k_eff, dtype=float) + 2.0)
    dcg = float((gains / discounts).sum())
    ideal_hits = _bi.min(k_eff, int((d["y"] > 0).sum()))
    if ideal_hits <= 0:
        return np.nan
    idcg = float((np.ones(ideal_hits, dtype=float) / np.log2(np.arange(ideal_hits, dtype=float) + 2.0)).sum())
    return dcg / idcg if idcg > 0 else np.nan


def stable_hash_text(text):
    return hashlib.md5(str(text).encode("utf-8")).hexdigest()


def month_ordinal(ts):
    t = pd.Timestamp(ts)
    return int(t.year * 12 + t.month)


def label_mode_tag(mode):
    if mode == "legacy_rebalance":
        return "legacy"
    if mode == "label_end_safe":
        return "safe"
    return str(mode).replace(" ", "_")


def train_start_for_policy(policy, cutoff):
    cutoff = pd.Timestamp(cutoff).normalize()
    if policy.get("train_window_months") is None:
        return pd.Timestamp("2019-01-01")
    months = int(policy["train_window_months"])
    return (cutoff - pd.DateOffset(months=months - 1)).replace(day=1)


def normalize_task(task):
    out = dict(task)
    out["train_start"] = pd.Timestamp(out["train_start"]).normalize()
    out["train_end"] = pd.Timestamp(out["train_end"]).normalize()
    out["test_start"] = (out["train_end"] + pd.DateOffset(months=1)).replace(day=1)
    return out


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = _bi.sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = _bi.sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = _bi.max(6, int(round(len(months) * 0.20)))
    n_valid = _bi.min(n_valid, len(months) - 1)
    valid_months = set(months[-n_valid:])
    fit = train_df[~train_df[DATE_COL].isin(valid_months)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_months)].copy()
    if len(fit) == 0 or len(valid) == 0:
        return train_df.copy(), train_df.copy()
    return fit, valid


def make_train_df(df_all, task):
    task = normalize_task(task)
    mode = task["label_boundary_mode"]
    mask = (df_all[DATE_COL] >= task["train_start"]) & (df_all[DATE_COL] <= task["train_end"])
    if mode == "label_end_safe":
        mask = mask & (df_all["next_date"] <= task["train_end"])
    elif mode != "legacy_rebalance":
        raise ValueError("unknown label boundary mode: " + str(mode))
    return df_all[mask].copy()


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=_bi.max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def needs_v4_adapter(feature_cols):
    adapter_cols = set(HYBRID_LIGHT_EXTRA_COLS)
    return _bi.any(c in adapter_cols for c in feature_cols)


## 5. 数据模块：加载、校验、压缩


In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, FACTOR_DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if FACTOR_DATE_COL not in df.columns and "feature_date" in df.columns:
        df[FACTOR_DATE_COL] = df["feature_date"]
    if FACTOR_DATE_COL not in df.columns:
        df[FACTOR_DATE_COL] = df[DATE_COL]
    if "next_date" not in df.columns:
        df["next_date"] = df[DATE_COL]
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = pd.to_numeric(df["raw_return_1m"], errors="coerce") - pd.to_numeric(df["benchmark_csi800_1m"], errors="coerce")
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, FACTOR_DATE_COL, "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    df = df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    return df


def audit_dataset(df):
    rows = []
    rows.append({"check": "rows", "value": int(len(df)), "status": "ok" if len(df) > 0 else "fail", "detail": ""})
    rows.append({"check": "date_min", "value": str(df[DATE_COL].min()), "status": "info", "detail": ""})
    rows.append({"check": "date_max", "value": str(df[DATE_COL].max()), "status": "info", "detail": ""})
    rows.append({"check": "months", "value": int(df[DATE_COL].nunique()), "status": "ok" if df[DATE_COL].nunique() >= MIN_TRAIN_MONTHS else "fail", "detail": ""})
    dup_count = int(df.duplicated([STOCK_COL, DATE_COL]).sum())
    rows.append({"check": "duplicate_stock_date", "value": dup_count, "status": "ok" if dup_count == 0 else "warn", "detail": "duplicates are kept last by downstream only if you fix input"})
    target_nan = int(df[TARGET_COL].isnull().sum())
    rows.append({"check": "target_nan", "value": target_nan, "status": "ok" if target_nan == 0 else "warn", "detail": ""})
    feature_missing = [c for c in FULL_V46_COLS if c not in df.columns]
    rows.append({"check": "full_v46_missing_features", "value": len(feature_missing), "status": "ok" if len(feature_missing) == 0 else "fail", "detail": ",".join(feature_missing)})
    audit_df = pd.DataFrame(rows)
    return audit_df



def add_industry_relative_features(df, candidate_cols):
    derived_cols = [c for c in candidate_cols if is_industry_relative_feature(c)]
    if len(derived_cols) == 0:
        return df
    out = df
    group_keys = [out[DATE_COL], out[INDUSTRY_COL]]
    created = []
    for dcol in progress_iter(derived_cols, total=len(derived_cols), desc="build industry-relative ranks"):
        base_col = dcol[len(DERIVED_PREFIX):]
        if base_col not in out.columns:
            continue
        vals = pd.to_numeric(out[base_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        ranked = vals.groupby(group_keys).rank(method="average", ascending=True)
        valid = vals.notnull().astype(np.int16)
        counts = valid.groupby(group_keys).transform("sum")
        denom = counts.where(counts > 0).astype(np.float32)
        out[dcol] = (ranked / denom).astype(np.float32)
        created.append(dcol)
        del vals, ranked, valid, counts, denom
        gc.collect()
    print("industry-relative features created:", len(created), created[:50])
    return out


def build_family_availability(df):
    rows = []
    usable = []
    for fam in FEATURE_FAMILIES:
        cols = []
        missing = []
        low_coverage = []
        for c in fam["candidate_cols"]:
            if c not in df.columns:
                missing.append(c)
                continue
            coverage = float(pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).notnull().mean())
            if coverage < MIN_FEATURE_NON_NULL_RATIO:
                missing.append(c)
                low_coverage.append(c)
            else:
                cols.append(c)
        cols = unique_keep_order(cols)
        row = {
            "family": fam["family"],
            "feature_variant": fam["feature_variant"],
            "description": fam["description"],
            "candidate_feature_count": len(fam["candidate_cols"]),
            "available_feature_count": len(cols),
            "missing_feature_count": len(missing),
            "low_coverage_feature_count": len(low_coverage),
            "available_features": ",".join(cols),
            "missing_features": ",".join(missing),
            "low_coverage_features": ",".join(low_coverage),
        }
        rows.append(row)
        min_cols = int(fam.get("min_available_cols", 20))
        if len(cols) >= min_cols and (not FAIL_ON_MISSING_FAMILY_FEATURES or len(missing) == 0):
            nf = dict(fam)
            nf["available_cols"] = cols
            nf["missing_cols"] = missing
            usable.append(nf)
        else:
            print("family not usable", fam["family"], "available", len(cols), "missing", len(missing), missing[:30])
    manifest = pd.DataFrame(rows)
    return manifest, usable

maybe_rebuild_dataset()
DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
data_audit_df = audit_dataset(df_all)
data_audit_df.to_csv(QA_DIR / "v82_data_audit.csv", index=False)

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())
print("next_date:", df_all["next_date"].min(), "->", df_all["next_date"].max())
display_df(data_audit_df, 40)
display_df(df_all[[TARGET_COL]].describe().T, 5)

fail_checks = data_audit_df[data_audit_df["status"] == "fail"]
if len(fail_checks):
    display_df(fail_checks, 20)
    raise ValueError("data audit failed; fix input data before training")
if FAIL_ON_MISSING_FULL_FEATURES:
    missing = [c for c in FULL_V46_COLS if c not in df_all.columns]
    if missing:
        raise ValueError("missing full_v46 features: " + ",".join(missing))

df_all = add_industry_relative_features(df_all, ALL_CANDIDATE_COLS)
family_availability_df, USABLE_FEATURE_FAMILIES = build_family_availability(df_all)
family_availability_df.to_csv(MANIFEST_DIR / "v82_family_availability.csv", index=False)
family_availability_df.to_csv(OUT_DIR / "v82_family_availability.csv", index=False)
display_df(family_availability_df[["family", "candidate_feature_count", "available_feature_count", "missing_feature_count", "low_coverage_feature_count", "missing_features"]], 40)
if len(USABLE_FEATURE_FAMILIES) == 0:
    raise ValueError("no usable feature family; check missing features or set REBUILD_DATA=True")
if FAIL_ON_MISSING_FAMILY_FEATURES and int(family_availability_df["missing_feature_count"].sum()) > 0:
    raise ValueError("selected feature families have missing features; set REBUILD_DATA=True or provide enriched CSV")

# Keep only production training columns to reduce memory.
panel_feature_cols = []
for fam in USABLE_FEATURE_FAMILIES:
    panel_feature_cols.extend(fam["available_cols"])
panel_feature_cols = unique_keep_order(panel_feature_cols)
keep_cols = unique_keep_order([STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, FACTOR_DATE_COL, "next_date", "raw_return_1m", "benchmark_csi800_1m"] + panel_feature_cols)
keep_cols = [c for c in keep_cols if c in df_all.columns]
df_all = df_all[keep_cols].copy()
for col in progress_iter(panel_feature_cols + [TARGET_COL], total=len(panel_feature_cols) + 1, desc="compact float32"):
    if col in df_all.columns and df_all[col].dtype == np.float64:
        df_all[col] = df_all[col].astype(np.float32)
gc.collect()
print("compacted:", df_all.shape)
print("usable families:", [x["family"] for x in USABLE_FEATURE_FAMILIES])


## 6. 训练计划生成


In [ ]:
def build_training_tasks(df):
    data_min = pd.Timestamp(df[DATE_COL].min()).normalize()
    data_max = pd.Timestamp(df[DATE_COL].max()).normalize()
    rows = []
    for fam in USABLE_FEATURE_FAMILIES:
        for mode in LABEL_BOUNDARY_MODES:
            for policy in TRAIN_POLICIES:
                for cutoff_str in TRAIN_CUTOFFS:
                    cutoff = pd.Timestamp(cutoff_str).normalize()
                    train_start = train_start_for_policy(policy, cutoff)
                    if train_start < data_min:
                        train_start = data_min
                    train_mask = (df[DATE_COL] >= train_start) & (df[DATE_COL] <= cutoff)
                    if mode == "label_end_safe":
                        train_mask = train_mask & (df["next_date"] <= cutoff)
                    elif mode != "legacy_rebalance":
                        raise ValueError("unknown label boundary mode: " + str(mode))
                    train_months = int(df.loc[train_mask, DATE_COL].nunique())
                    train_rows = int(train_mask.sum())
                    min_months = int(policy.get("min_train_months", MIN_TRAIN_MONTHS))
                    if train_months < min_months:
                        print("skip too few train months", fam["family"], mode, policy["train_policy"], cutoff, train_months)
                        continue
                    tag = "%s__%s__%s__cutoff%s" % (fam["family"], label_mode_tag(mode), policy["train_policy"], cutoff.strftime("%Y%m%d"))
                    rows.append({
                        "task_id": tag,
                        "family": fam["family"],
                        "feature_variant": fam["feature_variant"],
                        "label_boundary_mode": mode,
                        "train_policy": policy["train_policy"],
                        "method_type": policy["method_type"],
                        "train_window_months": policy.get("train_window_months"),
                        "train_start": train_start,
                        "train_end": cutoff,
                        "test_start": (cutoff + pd.DateOffset(months=1)).replace(day=1),
                        "train_months": train_months,
                        "train_rows_expected": train_rows,
                        "policy_description": policy.get("description", ""),
                        "data_min_date": data_min,
                        "data_max_date": data_max,
                    })
    task_df = pd.DataFrame(rows)
    if len(task_df):
        task_df = task_df.sort_values(["family", "label_boundary_mode", "train_policy", "train_end"]).reset_index(drop=True)
    if SMOKE_TEST and len(task_df):
        task_df = task_df.head(SMOKE_MAX_TASKS).copy()
    return task_df


training_task_df = build_training_tasks(df_all)
training_task_df.to_csv(MANIFEST_DIR / "v82_training_task_plan.csv", index=False)
print("training tasks:", training_task_df.shape)
display_df(training_task_df, 40)
if len(training_task_df) == 0:
    raise ValueError("empty training task plan")


## 7. 模型训练与 pkl 导出


In [ ]:
def make_model_id(task, feature_variant):
    return "%s_%s_%s_%s_%s_start%s_cutoff%s_fixed%s" % (
        MODEL_VERSION_PREFIX,
        task["family"],
        feature_variant,
        task["label_boundary_mode"],
        task["train_policy"],
        pd.Timestamp(task["train_start"]).strftime("%Y%m%d"),
        pd.Timestamp(task["train_end"]).strftime("%Y%m%d"),
        int(FIXED_ITER),
    )


def bundle_filename(model_id):
    return "model_%s.pkl" % model_id


def evaluate_holdout_model(model, diag_valid_df, feature_cols, fill_values):
    X_valid, y_valid, _, valid_index = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, fill_values)
    if len(X_valid) == 0:
        return [], np.nan
    pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    panel = diag_valid_df.loc[valid_index, [DATE_COL, STOCK_COL, TARGET_COL]].copy()
    panel["score"] = pred
    rows = []
    month_groups = panel.groupby(DATE_COL)
    for rebalance_date, month_df in progress_iter(month_groups, total=panel[DATE_COL].nunique(), desc="holdout monthly metrics", leave=False):
        month_df = month_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET_COL, "score"]).copy()
        n = int(len(month_df))
        if n < 3:
            continue
        true_n = _bi.min(int(HOLDOUT_TRUE_TOP_N), n)
        pred_k = _bi.min(int(HOLDOUT_PRED_TOP_K), n)
        true_index = set(month_df.sort_values(TARGET_COL, ascending=False).head(true_n).index)
        pred_df = month_df.sort_values("score", ascending=False).head(pred_k)
        pred_index = set(pred_df.index)
        hits = int(len(true_index.intersection(pred_index)))
        binary_true = month_df.index.to_series().isin(true_index).astype(int).values
        universe_alpha = float(month_df[TARGET_COL].mean())
        top8_alpha = float(pred_df[TARGET_COL].mean()) if len(pred_df) else np.nan
        rows.append({
            "rebalance_date": pd.Timestamp(rebalance_date),
            "n": n,
            "rank_ic": safe_rank_ic(month_df[TARGET_COL], month_df["score"]),
            "auc_true_top20": binary_rank_auc(binary_true, month_df["score"].values),
            "precision_at8_true20": hits / float(pred_k) if pred_k else np.nan,
            "recall_at8_true20": hits / float(true_n) if true_n else np.nan,
            "ndcg_at8_true20": binary_ndcg_at_k(binary_true, month_df["score"].values, pred_k),
            "top8_alpha": top8_alpha,
            "top8_edge": top8_alpha - universe_alpha if not pd.isnull(top8_alpha) else np.nan,
        })
    return rows, safe_rank_ic(y_valid, pred)


def export_model_bundle(task, trained, feature_cols, removed_cols, diag_rank_ic, train_df, variant):
    model_id = make_model_id(task, variant["feature_variant"])
    model_file = bundle_filename(model_id)
    model_path = MODEL_EXPORT_DIR / model_file
    feature_signature = stable_hash_text("|".join(feature_cols))
    data_signature = stable_hash_text("%s|%s|%s|%s|%s" % (str(DATA_PATH), len(df_all), df_all[DATE_COL].min(), df_all[DATE_COL].max(), TARGET_COL))
    fill_values = trained["fill_values"]
    if hasattr(fill_values, "to_dict"):
        fill_values = fill_values.to_dict()
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": model_id,
        "product_version": PRODUCT_VERSION,
        "run_name": RUN_NAME,
        "run_timestamp": RUN_TIMESTAMP,
        "benchmark": BENCHMARK,
        "model_family": MODEL_FAMILY,
        "family": task["family"],
        "feature_variant": variant["feature_variant"],
        "model_role": MODEL_ROLE,
        "label_boundary_mode": task["label_boundary_mode"],
        "require_label_end_within_train": bool(task["label_boundary_mode"] == "label_end_safe"),
        "legacy_unsealed_boundary": bool(task["label_boundary_mode"] == "legacy_rebalance"),
        "train_policy": task["train_policy"],
        "method_type": task["method_type"],
        "train_window_months": task.get("train_window_months"),
        "train_start": pd.Timestamp(task["train_start"]),
        "train_end": pd.Timestamp(task["train_end"]),
        "test_start": pd.Timestamp(task["test_start"]),
        "label_end": pd.Timestamp(task["train_end"]),
        "target_col": TARGET_COL,
        "target_note": "direct LGB regression on alpha_1m, fixed iteration, no early stopping",
        "data_file": str(DATA_PATH),
        "data_signature": data_signature,
        "feature_signature": feature_signature,
        "protocol": "v82_multi_family_production_model_export",
        "param_set": "v46_base_ff10_original",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(fill_values),
        "base_best_iter": int(FIXED_ITER),
        "model_iter": int(FIXED_ITER),
        "fixed_iter": int(FIXED_ITER),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(diag_rank_ic) if not pd.isnull(diag_rank_ic) else np.nan,
            "diag_definition": "mean monthly RankIC on final 20pct months using a fit-only diagnostic model",
        },
        "base_removed_features": list(removed_cols),
        "overlay_mode": "direct",
        "overlay_weight": 0.0,
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "portfolio_rule": PORTFOLIO_RULE,
        "board_caps": dict(BOARD_CAPS),
        "board_caps_text": BOARD_CAPS_TEXT,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": bool(needs_v4_adapter(feature_cols)),
        "requires_industry_relative_adapter": bool(_bi.any(is_industry_relative_feature(c) for c in feature_cols)),
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
        "train_row_count": int(len(train_df)),
        "train_month_count": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()) if "next_date" in train_df.columns else "",
        "note": "multi-family production training export; compatible with simplified JQ backtest loader when required features are available",
    }
    if EXPORT_MODELS:
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f, protocol=PICKLE_PROTOCOL)
    return model_id, model_file, model_path, bundle


def train_one_task(task, variant):
    task = normalize_task(task)
    train_df = make_train_df(df_all, task)
    if len(train_df) == 0:
        raise ValueError("empty train_df for task " + str(task.get("task_id")))
    if train_df[DATE_COL].nunique() < int(task.get("min_train_months", MIN_TRAIN_MONTHS)):
        raise ValueError("too few train months for task " + str(task.get("task_id")))

    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, variant.get("available_cols", variant["candidate_cols"]))

    holdout_rows = []
    diag_pooled_rank_ic = np.nan
    if ENABLE_HOLDOUT_DIAGNOSTICS:
        diagnostic = train_direct_lgb(diag_fit_df, feature_cols)
        holdout_rows, diag_pooled_rank_ic = evaluate_holdout_model(
            diagnostic["model"], diag_valid_df, feature_cols, diagnostic["fill_values"]
        )
        del diagnostic
        gc.collect()

    # Refit the exported production model on every eligible training row.
    trained = train_direct_lgb(train_df, feature_cols)
    diag_rank_ic = safe_mean([r["rank_ic"] for r in holdout_rows])
    model_id, model_file, model_path, bundle = export_model_bundle(
        task, trained, feature_cols, removed_cols, diag_rank_ic, train_df, variant
    )

    for r in holdout_rows:
        r["model_id"] = model_id
        r["family"] = task["family"]
        r["label_boundary_mode"] = task["label_boundary_mode"]
        r["train_policy"] = task["train_policy"]
        r["train_end"] = task["train_end"]

    importance_gain = np.asarray(trained["model"].feature_importance(importance_type="gain"), dtype=float)
    importance_split = np.asarray(trained["model"].feature_importance(importance_type="split"), dtype=float)
    gain_total = float(importance_gain.sum())
    importance_rows = []
    importance_iter = enumerate(feature_cols)
    for i, feature in progress_iter(importance_iter, total=len(feature_cols), desc="collect feature importance", leave=False):
        importance_rows.append({
            "model_id": model_id,
            "family": task["family"],
            "label_boundary_mode": task["label_boundary_mode"],
            "train_policy": task["train_policy"],
            "train_end": task["train_end"],
            "feature": feature,
            "importance_gain": float(importance_gain[i]),
            "importance_gain_pct": float(importance_gain[i] / gain_total) if gain_total > 0 else np.nan,
            "importance_split": float(importance_split[i]),
        })

    row = {
        "model_id": model_id,
        "model_file": model_file,
        "model_path": str(model_path),
        "family": task["family"],
        "feature_variant": variant["feature_variant"],
        "label_boundary_mode": task["label_boundary_mode"],
        "train_policy": task["train_policy"],
        "method_type": task["method_type"],
        "train_window_months": task.get("train_window_months"),
        "train_start": task["train_start"],
        "train_end": task["train_end"],
        "test_start": task["test_start"],
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "holdout_start": str(diag_valid_df[DATE_COL].min().date()) if len(diag_valid_df) else "",
        "holdout_end": str(diag_valid_df[DATE_COL].max().date()) if len(diag_valid_df) else "",
        "holdout_months": int(diag_valid_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()) if "next_date" in train_df.columns else "",
        "candidate_feature_count": len(variant["candidate_cols"]),
        "selected_feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "diag_pooled_rank_ic": diag_pooled_rank_ic,
        "holdout_auc_true_top20": safe_mean([r["auc_true_top20"] for r in holdout_rows]),
        "holdout_precision_at8_true20": safe_mean([r["precision_at8_true20"] for r in holdout_rows]),
        "holdout_recall_at8_true20": safe_mean([r["recall_at8_true20"] for r in holdout_rows]),
        "holdout_ndcg_at8_true20": safe_mean([r["ndcg_at8_true20"] for r in holdout_rows]),
        "holdout_top8_edge": safe_mean([r["top8_edge"] for r in holdout_rows]),
        "feature_signature": bundle["feature_signature"],
        "data_signature": bundle["data_signature"],
        "requires_v4_feature_adapter": bundle["requires_v4_feature_adapter"],
        "portfolio_rule": PORTFOLIO_RULE,
        "stock_num": STOCK_NUM,
        "board_caps": BOARD_CAPS_TEXT,
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    }
    del trained, train_df, diag_fit_df, diag_valid_df
    gc.collect()
    return row, importance_rows, holdout_rows


manifest_rows = []
feature_importance_rows = []
holdout_monthly_rows = []
variant_map = dict((fam["family"], fam) for fam in USABLE_FEATURE_FAMILIES)
for _, task_row in progress_iter(training_task_df.iterrows(), total=len(training_task_df), desc="train/export models"):
    task = task_row.to_dict()
    variant = variant_map[task["family"]]
    row, importance_one, holdout_one = train_one_task(task, variant)
    manifest_rows.append(row)
    feature_importance_rows.extend(importance_one)
    holdout_monthly_rows.extend(holdout_one)
    print("exported", row["model_file"], "family", row["family"], "rows", row["train_rows"], "features", row["selected_feature_count"], "holdout_ic", row["diag_rank_ic"])

model_manifest_df = pd.DataFrame(manifest_rows)
feature_importance_df = pd.DataFrame(feature_importance_rows)
holdout_monthly_df = pd.DataFrame(holdout_monthly_rows)
holdout_summary_cols = [
    "model_id", "family", "label_boundary_mode", "train_policy", "train_start", "train_end",
    "holdout_start", "holdout_end", "holdout_months", "diag_rank_ic", "diag_pooled_rank_ic",
    "holdout_auc_true_top20", "holdout_precision_at8_true20", "holdout_recall_at8_true20",
    "holdout_ndcg_at8_true20", "holdout_top8_edge",
]
holdout_summary_df = model_manifest_df[[c for c in holdout_summary_cols if c in model_manifest_df.columns]].copy()

model_manifest_df.to_csv(MANIFEST_DIR / "v82_model_manifest.csv", index=False)
model_manifest_df.to_csv(OUT_DIR / "v82_model_manifest.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v82_feature_importance.csv", index=False)
holdout_monthly_df.to_csv(OUT_DIR / "v82_holdout_monthly_metrics.csv", index=False)
holdout_summary_df.to_csv(OUT_DIR / "v82_holdout_summary.csv", index=False)
display_df(model_manifest_df, 40)
display_df(holdout_summary_df, 40)


## 8. 训练质量与模型解释可视化

这些图只展示数据质量、时间留出诊断、特征筛选稳定性和模型重要性。时间留出指标用于训练健康检查，不等同于完整 walk-forward OOS 或实盘收益。


In [ ]:
# =========================
# Data profiles and model diagnostics
# =========================
monthly_profile_rows = []
for rebalance_date, month_df in progress_iter(df_all.groupby(DATE_COL), total=df_all[DATE_COL].nunique(), desc="build monthly data profile"):
    y = pd.to_numeric(month_df[TARGET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    monthly_profile_rows.append({
        "rebalance_date": pd.Timestamp(rebalance_date),
        "sample_count": int(len(y)),
        "alpha_mean": float(y.mean()) if len(y) else np.nan,
        "alpha_std": float(y.std()) if len(y) else np.nan,
        "alpha_p05": float(y.quantile(0.05)) if len(y) else np.nan,
        "alpha_p95": float(y.quantile(0.95)) if len(y) else np.nan,
        "alpha_abs_gt20_rate": float((y.abs() > 0.20).mean()) if len(y) else np.nan,
    })
monthly_data_profile_df = pd.DataFrame(monthly_profile_rows).sort_values("rebalance_date")
monthly_data_profile_df.to_csv(QA_DIR / "v82_monthly_data_profile.csv", index=False)

candidate_union = []
for family in USABLE_FEATURE_FAMILIES:
    candidate_union.extend(family.get("available_cols", family["candidate_cols"]))
candidate_union = unique_keep_order([c for c in candidate_union if c in df_all.columns])
feature_quality_rows = []
for feature in progress_iter(candidate_union, total=len(candidate_union), desc="audit feature quality"):
    s = pd.to_numeric(df_all[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    feature_quality_rows.append({
        "feature": feature,
        "rows": int(len(s)),
        "valid_rows": int(len(valid)),
        "missing_rate": float(s.isnull().mean()),
        "unique_values": int(valid.nunique()) if len(valid) else 0,
        "p01": float(valid.quantile(0.01)) if len(valid) else np.nan,
        "median": float(valid.median()) if len(valid) else np.nan,
        "p99": float(valid.quantile(0.99)) if len(valid) else np.nan,
    })
feature_quality_df = pd.DataFrame(feature_quality_rows).sort_values(["missing_rate", "feature"], ascending=[False, True])
feature_quality_df.to_csv(QA_DIR / "v82_feature_quality.csv", index=False)

stability_rows = []
stability_group_cols = ["family", "label_boundary_mode", "train_policy"]
stability_groups = list(model_manifest_df.groupby(stability_group_cols))
for keys, group_df in progress_iter(stability_groups, total=len(stability_groups), desc="feature stability"):
    group_df = group_df.sort_values("train_end")
    records = group_df.to_dict("records")
    for i in range(1, len(records)):
        prev_row = records[i - 1]
        curr_row = records[i]
        prev_features = set([x for x in str(prev_row["feature_cols"]).split(",") if x])
        curr_features = set([x for x in str(curr_row["feature_cols"]).split(",") if x])
        union = prev_features.union(curr_features)
        overlap = prev_features.intersection(curr_features)
        stability_rows.append({
            "family": keys[0],
            "label_boundary_mode": keys[1],
            "train_policy": keys[2],
            "previous_train_end": prev_row["train_end"],
            "current_train_end": curr_row["train_end"],
            "previous_feature_count": len(prev_features),
            "current_feature_count": len(curr_features),
            "intersection_count": len(overlap),
            "union_count": len(union),
            "feature_jaccard": len(overlap) / float(len(union)) if len(union) else np.nan,
        })
feature_stability_df = pd.DataFrame(stability_rows)
feature_stability_df.to_csv(OUT_DIR / "v82_adjacent_cutoff_feature_stability.csv", index=False)


def short_model_label(row):
    cutoff = pd.Timestamp(row["train_end"]).strftime("%Y%m")
    policy = "exp" if str(row["train_policy"]).startswith("expanding") else str(row["train_policy"])
    mode = "safe" if row["label_boundary_mode"] == "label_end_safe" else "legacy"
    return "%s|%s|%s|%s" % (row["family"], policy, cutoff, mode)


def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(str(path), dpi=PLOT_DPI, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print("saved figure:", path)


# Figure 1: data coverage and label behavior.
if len(monthly_data_profile_df):
    p = monthly_data_profile_df
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(p["rebalance_date"], p["sample_count"], color="#285f9e", linewidth=1.8)
    axes[0].set_title("Monthly sample coverage")
    axes[0].set_ylabel("stocks")
    axes[1].plot(p["rebalance_date"], p["alpha_std"], color="#d4543c", linewidth=1.8, label="std")
    axes[1].fill_between(p["rebalance_date"].values, p["alpha_p05"].values, p["alpha_p95"].values, color="#7aa66c", alpha=0.24, label="p05-p95")
    axes[1].axhline(0, color="#555555", linewidth=0.8)
    axes[1].set_title("Monthly alpha dispersion")
    axes[1].legend(loc="upper right")
    axes[2].plot(p["rebalance_date"], p["alpha_abs_gt20_rate"], color="#8c5bb3", linewidth=1.8)
    axes[2].set_title("Share of labels with absolute alpha above 20%")
    axes[2].set_ylabel("rate")
    save_figure(fig, "v82_data_coverage_and_label_profile.png")


# Figure 2: honest temporal holdout metrics.
if len(model_manifest_df):
    plot_df = model_manifest_df.sort_values(["family", "label_boundary_mode", "train_policy", "train_end"]).copy()
    labels = [short_model_label(r) for _, r in plot_df.iterrows()]
    metrics = [
        ("diag_rank_ic", "Holdout monthly RankIC", 0.0),
        ("holdout_auc_true_top20", "AUC: true Top20", 0.5),
        ("holdout_precision_at8_true20", "Precision@8: true Top20", None),
        ("holdout_recall_at8_true20", "Recall@8: true Top20", None),
        ("holdout_ndcg_at8_true20", "NDCG@8: true Top20", None),
        ("holdout_top8_edge", "Top8 alpha edge", 0.0),
    ]
    colors = ["#285f9e", "#d4543c", "#2d8f78", "#e39c24", "#7b61a8", "#59636d"]
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    positions = np.arange(len(plot_df))
    for i, item in progress_iter(enumerate(metrics), total=len(metrics), desc="plot holdout metrics", leave=False):
        col, title, baseline = item
        ax = axes.flat[i]
        values = pd.to_numeric(plot_df[col], errors="coerce").values
        ax.bar(positions, values, color=colors[i], width=0.75)
        if baseline is not None:
            ax.axhline(baseline, color="#333333", linewidth=1.0, linestyle="--")
        ax.set_title(title)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=8)
    save_figure(fig, "v82_holdout_metrics_dashboard.png")


# Figure 3: fit versus holdout and feature count.
if len(model_manifest_df):
    plot_df = model_manifest_df.sort_values(["family", "train_policy", "train_end"]).copy()
    labels = [short_model_label(r) for _, r in plot_df.iterrows()]
    positions = np.arange(len(plot_df))
    width = 0.38
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    axes[0].bar(positions - width / 2.0, plot_df["train_rank_ic"].values, width=width, color="#285f9e", label="production fit IC")
    axes[0].bar(positions + width / 2.0, plot_df["diag_rank_ic"].values, width=width, color="#d4543c", label="temporal holdout IC")
    axes[0].axhline(0, color="#333333", linewidth=0.8)
    axes[0].set_title("Fit versus temporal holdout RankIC")
    axes[0].legend(loc="best")
    axes[1].bar(positions, plot_df["selected_feature_count"].values, color="#2d8f78", label="selected")
    axes[1].bar(positions, plot_df["removed_feature_count"].values, bottom=plot_df["selected_feature_count"].values, color="#e39c24", label="correlation-pruned")
    axes[1].set_title("Feature selection inventory")
    axes[1].legend(loc="best")
    for ax in axes:
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=70, ha="right", fontsize=8)
    save_figure(fig, "v82_fit_holdout_and_feature_inventory.png")


# Figure 4: aggregate gain importance, one panel per family.
if len(feature_importance_df):
    families = _bi.sorted(feature_importance_df["family"].dropna().unique())
    fig, axes = plt.subplots(1, len(families), figsize=(7 * len(families), 8), squeeze=False)
    family_colors = ["#285f9e", "#2d8f78", "#d4543c"]
    for i, family in progress_iter(enumerate(families), total=len(families), desc="plot feature importance", leave=False):
        part = feature_importance_df[feature_importance_df["family"] == family]
        gain = part.groupby("feature")["importance_gain_pct"].mean().sort_values(ascending=False).head(15).sort_values()
        axes[0, i].barh(np.arange(len(gain)), gain.values, color=family_colors[i % len(family_colors)])
        axes[0, i].set_yticks(np.arange(len(gain)))
        axes[0, i].set_yticklabels(gain.index, fontsize=9)
        axes[0, i].set_title("%s: mean gain share" % family)
        axes[0, i].set_xlabel("gain share")
    save_figure(fig, "v82_feature_importance_by_family.png")


# Figure 5: feature missingness and adjacent-cutoff stability.
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
if len(feature_quality_df):
    missing = feature_quality_df.sort_values("missing_rate", ascending=False).head(15).sort_values("missing_rate")
    axes[0].barh(np.arange(len(missing)), missing["missing_rate"].values, color="#d4543c")
    axes[0].set_yticks(np.arange(len(missing)))
    axes[0].set_yticklabels(missing["feature"], fontsize=9)
    axes[0].set_title("Highest feature missing rates")
    axes[0].set_xlabel("missing rate")
else:
    axes[0].set_axis_off()
if len(feature_stability_df):
    stability_plot = feature_stability_df.copy().reset_index(drop=True)
    stability_labels = ["%s|%s|%s" % (r["family"], r["train_policy"], pd.Timestamp(r["current_train_end"]).strftime("%Y%m")) for _, r in stability_plot.iterrows()]
    positions = np.arange(len(stability_plot))
    axes[1].bar(positions, stability_plot["feature_jaccard"].values, color="#7b61a8")
    axes[1].set_xticks(positions)
    axes[1].set_xticklabels(stability_labels, rotation=70, ha="right", fontsize=8)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("Adjacent-cutoff selected-feature Jaccard")
else:
    axes[1].text(0.5, 0.5, "Need at least two cutoffs", ha="center", va="center")
    axes[1].set_axis_off()
save_figure(fig, "v82_feature_quality_and_stability.png")

print("visual diagnostic outputs:")
for path in [
    QA_DIR / "v82_monthly_data_profile.csv",
    QA_DIR / "v82_feature_quality.csv",
    OUT_DIR / "v82_holdout_monthly_metrics.csv",
    OUT_DIR / "v82_holdout_summary.csv",
    OUT_DIR / "v82_feature_importance.csv",
    OUT_DIR / "v82_adjacent_cutoff_feature_stability.csv",
]:
    print("-", path)


## 9. 上传清单与完整性检查


In [ ]:
upload_rows = []
for _, r in model_manifest_df.iterrows():
    upload_rows.append({
        "model_id": r["model_id"],
        "model_file": r["model_file"],
        "local_model_path": r["model_path"],
        "joinquant_file_hint": "v4模型/csi800_ml_v82_full_v46_model_trainer_outputs/model_exports/" + r["model_file"],
        "family": r["family"],
        "label_boundary_mode": r["label_boundary_mode"],
        "train_policy": r["train_policy"],
        "train_start": r["train_start"],
        "train_end": r["train_end"],
        "selected_feature_count": r["selected_feature_count"],
        "train_rank_ic": r["train_rank_ic"],
        "diag_rank_ic": r["diag_rank_ic"],
        "portfolio_rule": r["portfolio_rule"],
        "stock_num": r["stock_num"],
        "board_caps": r["board_caps"],
    })
upload_list_df = pd.DataFrame(upload_rows)
upload_list_df.to_csv(MANIFEST_DIR / "v82_joinquant_upload_list.csv", index=False)
upload_list_df.to_csv(OUT_DIR / "v82_joinquant_upload_list.csv", index=False)

required_bundle_keys = [
    "objective", "product_version", "base_model", "base_feature_cols", "base_fill_values",
    "overlay_mode", "family", "train_policy", "label_boundary_mode", "top_n_candidates",
    "stock_num", "portfolio_rule", "requires_v4_feature_adapter",
]
check_rows = []
if RELOAD_CHECK:
    for _, r in progress_iter(model_manifest_df.iterrows(), total=len(model_manifest_df), desc="reload exported pkl"):
        path = Path(r["model_path"])
        row = {"model_file": r["model_file"], "path": str(path), "exists": path.exists(), "ok": False, "error": ""}
        try:
            with open(path, "rb") as f:
                bundle = pickle.load(f)
            missing = [k for k in required_bundle_keys if k not in bundle]
            row["missing_keys"] = ",".join(missing)
            row["ok"] = len(missing) == 0
            row["feature_count"] = len(bundle.get("base_feature_cols", []))
            row["overlay_mode"] = bundle.get("overlay_mode")
            row["family"] = bundle.get("family")
            row["train_policy"] = bundle.get("train_policy")
            row["label_boundary_mode"] = bundle.get("label_boundary_mode")
            row["requires_v4_feature_adapter"] = bundle.get("requires_v4_feature_adapter")
        except Exception as err:
            row["error"] = str(err)
        check_rows.append(row)
reload_check_df = pd.DataFrame(check_rows)
reload_check_df.to_csv(QA_DIR / "v82_reload_check.csv", index=False)

if len(reload_check_df) and not bool(reload_check_df["ok"].all()):
    display_df(reload_check_df, 40)
    raise ValueError("reload check failed")

print("saved outputs:")
for path in [
    OUT_DIR / "v82_model_manifest.csv",
    OUT_DIR / "v82_joinquant_upload_list.csv",
    QA_DIR / "v82_data_audit.csv",
    QA_DIR / "v82_reload_check.csv",
]:
    print("-", path)
print("model files:")
for _, r in model_manifest_df.iterrows():
    print("-", r["model_file"])

display_df(upload_list_df, 40)
display_df(reload_check_df, 40)


## 10. 输出说明


In [ ]:
readme_lines = []
readme_lines.append("# V82 full_v46 production model trainer outputs")
readme_lines.append("")
readme_lines.append("Run timestamp: %s" % RUN_TIMESTAMP)
readme_lines.append("Data path: %s" % DATA_PATH)
readme_lines.append("Label boundary modes: %s" % ",".join(LABEL_BOUNDARY_MODES))
readme_lines.append("Train policies: %s" % ",".join([x["train_policy"] for x in TRAIN_POLICIES]))
readme_lines.append("")
readme_lines.append("## Files")
readme_lines.append("- model_exports/*.pkl: upload these pkl files to JoinQuant and set PRIMARY_MODEL_FILE in the backtest.")
readme_lines.append("- v82_model_manifest.csv: training metadata and diagnostics.")
readme_lines.append("- v82_joinquant_upload_list.csv: concise upload checklist.")
readme_lines.append("- qa/v82_data_audit.csv: input data checks.")
readme_lines.append("- qa/v82_reload_check.csv: pkl reload checks.")
readme_lines.append("- v82_holdout_summary.csv and v82_holdout_monthly_metrics.csv: honest temporal holdout ranking diagnostics.")
readme_lines.append("- v82_feature_importance.csv and v82_adjacent_cutoff_feature_stability.csv: model explanation and feature-selection stability.")
readme_lines.append("- figures/*.png: data quality, holdout metrics, fit gap, importance and stability dashboards.")
readme_lines.append("")
readme_lines.append("## Families")
readme_lines.append("Default RUN_FAMILIES exports full_v46 only. Set RUN_FAMILIES to defensive_reversal_quality, balanced_style_plus, or all three for model-bank exports.")
readme_lines.append("")
readme_lines.append("## Data rebuild")
readme_lines.append("Set REBUILD_DATA=True in JoinQuant research to regenerate the CSI800 monthly training panel before model training.")
readme_lines.append("Set REBUILD_DATA_END_FOR_LABEL to an explicit date for reproducible data snapshots; None uses runtime today.")
readme_lines.append("")
readme_lines.append("## Diagnostics")
readme_lines.append("diag_rank_ic is the mean monthly RankIC from a model trained only before the final 20% holdout months; the exported production model is then refit on all eligible rows.")
readme_lines.append("Holdout diagnostics are training health checks, not a substitute for walk-forward OOS or JoinQuant execution backtests.")
readme_lines.append("")
readme_lines.append("## Default recommendation")
readme_lines.append("Use full_v46 + expanding_min36 + legacy_rebalance as mainline; use rolling60m as challenger/shadow.")

readme_path = OUT_DIR / "README_v82_model_trainer.md"
with open(readme_path, "w") as f:
    f.write("\n".join(readme_lines))
print("README:", readme_path)
print("done")
